# 1. Importacion de librerias
Se importan las libreias necesarias

In [8]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from copy import deepcopy

# 2. Funciones y clases necesarias para el AG

Cromosoma: [alpha,  batch, phi, rho]



### Se colocan las constantes de los anelos permitidos por gen

Rangos:
-   alpha = [1e-4, 1e-2]
-   batch = {8, 16, 32, 64}
-   phi = {adam, RMSprop, SGD}
-   rho = [0.0, 0.5]

In [1]:
ALPHA = (1e-4, 1e-2)
BATCH = (8, 16, 32, 64)
PHI = ("adam", "sgd", "rmsprop")
RHO = (0.0, 0.50)

### Se definen constantes utiles para las funciones

In [ ]:
# Tipo de cruzamiento por hiperparámetro:
UNIFORM_CROSS = ('batch', 'phi')
BLEND_CROSSOVER = ('alpha', 'rho')

# Probabilidad de mutacion:
MUTATION_RATE = 0.15

# Generaciones maximas
GEN_MAX = 80

# Paciencia en las generaciones
GEN_PAT = 10

# Individuos base en la seleccion
K_SELECT = 4

# Inicializacion con semilla
SEED = 29
random.seed(SEED)

### Definicion de la clase del cromosoma
Se crea una clase la cual contendra los hiperparametros del entrenamiento

In [ ]:
class HyperParams:
    def __init__(
        self,
        alpha: float|None = None,
        batch: int|None = None,
        phi: str|None = None,
        rho: float|None = None
    ):
        self.alpha = alpha
        self.batch = batch
        self.phi = phi
        self.rho = rho

    def __getitem__(
        self, 
        key: str,
    ):
        return getattr(self, key)
    
    def __setitem__(
        self, 
        key: str, 
        value,
    ):
        setattr(self, key, value)

    def keys(
        self,
    ):
        return ['alpha', 'batch', 'phi', 'rho']

    def copy(
        self,
    ):
        return deepcopy(self)


### Definicion de la estrutura del individuo
Se implementa una clase para cada invidiuo, la cual guarde su cromosoma, fitness y tenga metodos para el cruzamiento y mutaciones.

In [ ]:
class Individual:
    def __init__(
        self,
        chromosome: HyperParams|None = None,
    ):
        if chromosome is not None:
            self.chromosome = chromosome.copy()
        else:
            raise ValueError("Error: chromosome es nulo")
        
        self.fitness: float = -1

    def crossover(
        self,
        other,
    ):
        chromosome1 = HyperParams()
        chromosome2 = HyperParams()

        # Se hace un cruzamiento uniforme para los genes phi y batch:
        for param in UNIFORM_CROSS:
            if random.uniform(0, 1) <= 0.5:
                chromosome1[param] = self.chromosome[param]
                chromosome2[param] = other.chromosome[param]
            else:
                chromosome1[param] = other.chromosome[param]
                chromosome2[param] = self.chromosome[param]

        # Se hace un cruzamiento blend para los genes alpha y rho:
        for param in BLEND_CROSSOVER:
            alpha = random.uniform(0, 1)
            chromosome1[param] = alpha * self.chromosome[param] + (1 - alpha) * other.chromosome[param]
            chromosome2[param] = alpha * other.chromosome[param] + (1 - alpha) * self.chromosome[param]

        return Individual(chromosome1), Individual(chromosome2)
    
    def mutation(
        self,
    ):
        chromosome = self.chromosome.copy()

        for param in chromosome.keys():
            if random.uniform(0, 1) <= MUTATION_RATE:
                if param == 'alpha':
                    chromosome[param] = random.uniform(*ALPHA)
                elif param == 'batch':
                    chromosome[param] = random.choice(BATCH)
                elif param == 'phi':
                    chromosome[param] = random.choice(PHI)
                elif param == 'rho':
                    chromosome[param] = random.uniform(*RHO)

        return Individual(chromosome)

### Funcion para obtener el fitness de un cromosoma

In [ ]:
def get_fitness(
    chromosome: HyperParams,

) -> float:
    #TODO xd
    # Despues de tener el entrenamiento de las cnn

### Funcion para evaluar una poblacion de individuos

In [ ]:
def eval_population(
    population: list[Individual],
):
    pop_size = len(population)

    for i in range(pop_size):
        if population[i].fitness == -1:
            population[i].fitness = get_fitness(population[i].chromosome)

### Funcion para inicializar una poblacion de individuos


In [ ]:
def init_population(
    size: int, 
):
    population = []
    chromosome_already_used = set()
    for _ in range(size):
        alpha = random.uniform(*ALPHA)
        batch = random.choice(BATCH)
        phi = random.choice(PHI)
        rho = random.uniform(*RHO)

        chromosome = (alpha, batch, phi, rho)
        population.append(Individual(HyperParams(*chromosome)))
    
    return population


### Funcion para la seleccion de padres

In [ ]:
def select_parents(
    population: list[Individual],
    tournament_size: int,
):
    list_indiv = []

    x1 = np.random.permutation(len(population))
    y1 = x1[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y1[i]].fitness)

    iParent1 = np.argmax(list_indiv)

    list_indiv = []
    x2 = np.delete(x1, iParent1)
    x2 = np.random.permutation(x2)
    y2 = x2[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y2[i]].fitness)

    iParent2 = np.argmax(list_indiv)

    return population[y1[iParent1]], population[y2[iParent2]]

### Funcion para la seleccion de la poblacion

In [ ]:
def select_survivors(
    population: list[Individual],
    offspring: list[Individual],
    num_survivors: int,
):
    next_population = []
    population.extend(offspring)
    population.sort(key=lambda x: x.fitness, reverse=True)

    next_population = population[:num_survivors]
    
    return next_population

    

### Algoritmo genetico para encontrar soluciones

In [ ]:
def genetic_algo(
    population,
    n_gen = GEN_MAX,
    p_gen = GEN_PAT,
):
    
    pop_size = len(population)

    eval_population(population)

    best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
    best_fitness = [best.fitness]

    print(f"Poblacion inicial, mejor fitness = {best_fitness[:1]}")

    for gen in range(n_gen):

        mating_pool = []

        for i in range(int(pop_size/2)):
            mating_pool.append(select_parents(
                population=population,
                tournament_size=K_SELECT,
            ))
        
        offspring = []
        
        for i in range(int(pop_size/2)):
            papa = mating_pool[i][0]
            mama = mating_pool[i][1]
            offspring.extend(papa.crossover(
                other=mama
            ))

        offspring = [child.mutation() for child in offspring]

        eval_population(
            population=offspring
        )

        population = select_survivors(
            population=population,
            offspring=offspring,
            num_survivors=pop_size,
        )

        best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
        best_fitness.append(best.fitness)

        if best_fitness[-1] > best_fitness[-2]:
            print(f"Generación {gen}, mejor fitness = {best_fitness[-1]}")

    print(f"Mejor individuo en la ultima geneacion con fitness = {best_fitness[-1]}")
    return best, best_fitness

     